アップロード（返信された分割ファイルを結合する）全店舗変身されているかは要チェック

In [1]:
import pandas as pd
import io
from google.colab import files
from IPython.display import display

print('Excelファイルを選択してください（複数選択可）')
uploaded = files.upload()

Excelファイルを選択してください（複数選択可）


Saving 千葉事務所_健康診断対象者一覧.xlsx to 千葉事務所_健康診断対象者一覧.xlsx
Saving 平出事務所_健康診断対象者一覧.xlsx to 平出事務所_健康診断対象者一覧.xlsx
Saving 問屋_健康診断対象者一覧-2.xlsx to 問屋_健康診断対象者一覧-2.xlsx


セル2結合

In [2]:
import re

def read_meibo(filename, content):
    buf = io.BytesIO(content)
    df_raw = pd.read_excel(buf, sheet_name=0, header=None)
    buf.seek(0)
    if df_raw.iloc[0].isna().all():
        df = pd.read_excel(buf, sheet_name=0, header=1)
    else:
        df = pd.read_excel(buf, sheet_name=0, header=0)
    df = df.dropna(axis=1, how='all')
    df = df.dropna(how='all')
    df.insert(0, 'ファイル名', filename)
    return df

RENAME_MAP = {
    '事業所1':            '部門1名',
    '診断種別':           '健診内容',
    '前年度診断機関':     '前年度受診期間',
    '当年度受診予定機関': '今年度予約予定医療機関',
    '受診機関':           '受診時間',
}

TARGET_COLS = [
    'ファイル名', 'No', '従業員番号', '従業員名', '従業員名カナ',
    'エリア', '部門1名', '入社日', '生年月日', '性別', '年齢',
    '健康保険記号', '健康保険 - 被保険者整理番号',
    '健診内容', '前年度受診期間', '今年度予約予定医療機関',
    '受診日', '受診時間', 'オプション',
    '会社用送付先', '問診票送付先', '支払方法', '金額',
    'キャンセル済', '再申込済', '予約完了', '日付確定', '金額.1', '履歴', '受診時満年齢',
]

all_df = []
for filename, content in uploaded.items():
    df = read_meibo(filename, content)
    df = df.rename(columns=RENAME_MAP)
    all_df.append(df)

meibo = pd.concat(all_df, ignore_index=True, join='outer')
meibo['No'] = range(1, len(meibo) + 1)

# オプション列がない場合は空欄で作成
if 'オプション' not in meibo.columns:
    meibo['オプション'] = None

# オプションに"退職"が含まれる行は部門1名の先頭に"_退職"を追記
mask_retire = meibo['オプション'].astype(str).str.contains('退職', na=False)
meibo.loc[mask_retire, '部門1名'] = '退職_' + meibo.loc[mask_retire, '部門1名'].astype(str)

# オプションに"異動"が含まれる行は部門1名を異動先に書き換え
def extract_destination(val):
    s = str(val).strip()
    m = re.search(r'(.+?)[にへ]異動', s)
    if m: return m.group(1).strip()
    m = re.search(r'異動済(.+?)(?:へ)?$', s)
    if m: return m.group(1).strip()
    m = re.search(r'(.+?)異動済', s)
    if m: return m.group(1).strip()
    return None

mask_move = meibo['オプション'].astype(str).str.contains('異動', na=False)


destinations = meibo.loc[mask_move, 'オプション'].apply(extract_destination)
meibo.loc[mask_move, '部門1名'] = destinations

# 受診時満年齢を計算
def calc_age(row):
    try:
        bd = pd.to_datetime(row['生年月日'])
        rd = pd.to_datetime(row['受診日'])
        age = rd.year - bd.year
        if (rd.month, rd.day) < (bd.month, bd.day):
            age -= 1
        return age
    except:
        return None

meibo['受診時満年齢'] = meibo.apply(calc_age, axis=1)

# 列を TARGET_COLS 順に並べ替え（存在しない列は空欄で追加）
for col in TARGET_COLS:
    if col not in meibo.columns:
        meibo[col] = None
meibo = meibo[TARGET_COLS]

print(f'📋 合計: {len(meibo)} 件')
display(meibo)

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, numbers

def apply_styles(filepath):
    wb = load_workbook(filepath)
    ws = wb.active

    yellow = PatternFill(fill_type='solid', fgColor='FFD700')
    gray   = PatternFill(fill_type='solid', fgColor='D9D9D9')

    headers = {cell.value: cell.column for cell in ws[1]}
    col_age  = headers.get('受診時満年齢')
    col_opt  = headers.get('オプション')

    # 日付列を yyyy/mm/dd 形式に
    date_cols = ['生年月日', '受診日', '入社日']
    date_col_indices = [headers[c] for c in date_cols if c in headers]

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        opt_val = row[col_opt - 1].value if col_opt else None
        age_val = row[col_age - 1].value if col_age else None

        # オプションに値がある行 → 薄いグレー
        if opt_val not in (None, '', 'None', 'nan'):
            for cell in row:
                cell.fill = gray

        # 受診時満年齢75歳以上 → 濃い黄色
        if col_age and isinstance(age_val, (int, float)) and age_val >= 75:
            row[col_age - 1].fill = yellow

        # 日付列の書式
        for ci in date_col_indices:
            row[ci - 1].number_format = 'YYYY/MM/DD'

    wb.save(filepath)


📋 合計: 45 件


,ファイル名,No,従業員番号,従業員名,従業員名カナ,エリア,部門1名,入社日,生年月日,性別,...,問診票送付先,支払方法,金額,キャンセル済,再申込済,予約完了,日付確定,金額.1,履歴,受診時満年齢
0,千葉事務所_健康診断対象者一覧.xlsx,1,HH000241,佐々木 友美,ササキ トモミ,宇都宮,千葉事務所,2021-05-01,1977-09-04,女性,...,None,None,None,None,None,None,None,None,None,NaN
1,千葉事務所_健康診断対象者一覧.xlsx,2,HH000441,川中 美由紀,カワナカ ミユキ,千葉,千葉事務所,2022-03-14,1975-03-31,女性,...,None,None,None,None,None,None,None,None,None,NaN
2,平出事務所_健康診断対象者一覧.xlsx,3,HL000030,田村 昭美,タムラ アキミ,宇都宮,平出事務所,2015-06-01,1968-01-03,女性,...,None,None,None,None,None,None,None,None,None,55.0
3,平出事務所_健康診断対象者一覧.xlsx,4,HL000200,三浦 絵里香,ミウラ エリカ,宇都宮,平出事務所,2017-04-10,1973-06-20,女性,...,None,None,None,None,None,None,None,None,None,NaN
4,平出事務所_健康診断対象者一覧.xlsx,5,HL000265,村上 薫,ムラカミ ユキ,宇都宮,平出事務所,2017-08-01,1972-04-28,女性,...,None,None,None,None,None,None,None,None,None,NaN
5,平出事務所_健康診断対象者一覧.xlsx,6,HL000287,石田 佳代,イシダ カヨ,宇都宮,平出事務所,2017-10-16,1979-02-07,女性,...,None,None,None,None,None,None,None,None,None,NaN
6,平出事務所_健康診断対象者一覧.xlsx,7,HL000291,橋本 益子,ハシモト マスコ,宇都宮,平出事務所,2017-10-23,1980-10-15,女性,...,None,None,None,None,None,None,None,None,None,37.0
7,平出事務所_健康診断対象者一覧.xlsx,8,HL000613,佐藤 晴美,サトウ ハルミ,宇都宮,平出事務所,2019-04-15,1983-03-11,女性,...,None,None,None,None,None,None,None,None,None,44.0
8,平出事務所_健康診断対象者一覧.xlsx,9,HL001481,井上 祐子,イノウエ ユウコ,宇都宮,平出事務所,2022-06-13,1981-08-28,女性,...,None,None,None,None,None,None,None,None,None,36.0
9,平出事務所_健康診断対象者一覧.xlsx,10,HL001685,松嶋 望,マツシマ ノゾミ,宇都宮,退職_平出事務所,2023-01-16,1983-01-18,女性,...,None,None,None,None,None,None,None,None,None,NaN


セル3ダウンロード

In [3]:
output_path = '健康診断名簿_統合.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl', datetime_format='YYYY/MM/DD') as writer:
    meibo.to_excel(writer, index=False, sheet_name='名簿')
apply_styles(output_path)
files.download(output_path)
print(f'💾 {output_path} をダウンロードしました')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

💾 健康診断名簿_統合.xlsx をダウンロードしました


正しい手順：

セルAとセルBは必ずセットで1回ずつ実行してください。

セルA実行 → セルB実行   （1施設目追加）
セルA実行 → セルB実行   （2施設目追加）
セルA実行 → セルB実行   （3施設目追加）

【まとめて複数ファイルを追加したい場合】：
セルAの「追加ファイル選択」ダイアログで複数ファイルを一度に選択すればセルA→Bの1セットで済みます（Ctrlキーで複数選択可能）。

セルA

In [ ]:
print('① 既存の「健康診断名簿_統合.xlsx」を選択してください')
uploaded_base = files.upload()

print('② 追加したいExcelファイルを選択してください（複数可）')
uploaded_new = files.upload()

① 既存の「健康診断名簿_統合.xlsx」を選択してください


Saving 健康診断名簿_統合.xlsx to 健康診断名簿_統合.xlsx
② 追加したいExcelファイルを選択してください（複数可）


Saving 012GH甲府_健康診断対象者一覧.xlsx to 012GH甲府_健康診断対象者一覧.xlsx
Saving 058KM城東_健康診断対象者一覧.xlsx to 058KM城東_健康診断対象者一覧.xlsx
Saving 061KPつくば_健康診断対象者一覧.xlsx to 061KPつくば_健康診断対象者一覧.xlsx
Saving 068KP伊勢崎店_健康診断対象者一覧.xlsx to 068KP伊勢崎店_健康診断対象者一覧.xlsx
Saving 076KP西城南店プラス_健康診断対象者一覧 (2).xlsx to 076KP西城南店プラス_健康診断対象者一覧 (2).xlsx
Saving 078KP土浦_健康診断対象者一覧.xlsx to 078KP土浦_健康診断対象者一覧.xlsx
Saving 080KP学園の森_健康診断対象者一覧.xlsx to 080KP学園の森_健康診断対象者一覧.xlsx
Saving 084KP麗澤大学前_健康診断対象者一覧(1).xlsx to 084KP麗澤大学前_健康診断対象者一覧(1).xlsx
Saving 087KP上尾_健康診断対象者一覧.xlsx to 087KP上尾_健康診断対象者一覧.xlsx
Saving 102KPせんげん台_健康診断対象者一覧.xlsx to 102KPせんげん台_健康診断対象者一覧.xlsx
Saving 166KM郡山静西_健康診断対象者一覧 (1).xlsx to 166KM郡山静西_健康診断対象者一覧 (1).xlsx
Saving 173KP八千代村上_健康診断対象者一覧(1).xlsx to 173KP八千代村上_健康診断対象者一覧(1).xlsx


セルB

In [ ]:
import re

# 既存の統合ファイルを読み込む
base_filename = list(uploaded_base.keys())[0]
meibo_base = pd.read_excel(io.BytesIO(uploaded_base[base_filename]), sheet_name='名簿')
print(f'既存データ: {len(meibo_base)} 件')

# 追加ファイルを処理
add_df = []
for filename, content in uploaded_new.items():
    df = read_meibo(filename, content)
    df = df.rename(columns=RENAME_MAP)
    add_df.append(df)
    print(f'✅ {filename} → {len(df)} 件')

meibo_add = pd.concat(add_df, ignore_index=True, join='outer')

if 'オプション' not in meibo_add.columns:
    meibo_add['オプション'] = None

mask_retire = meibo_add['オプション'].astype(str).str.contains('退職', na=False)
meibo_add.loc[mask_retire, '部門1名'] = '退職_' + meibo_add.loc[mask_retire, '部門1名'].astype(str)

def extract_destination(val):
    s = str(val).strip()
    m = re.search(r'(.+?)[にへ][異移]動', s)
    if m: return m.group(1).strip()
    m = re.search(r'[異移]動済(.+?)(?:へ)?$', s)
    if m: return m.group(1).strip()
    m = re.search(r'(.+?)[異移]動済', s)
    if m: return m.group(1).strip()
    return None

mask_move = meibo_add['オプション'].astype(str).str.contains('[異移]動', na=False)
meibo_add.loc[mask_move, '部門1名'] = meibo_add.loc[mask_move, 'オプション'].apply(extract_destination)

def calc_age(row):
    try:
        bd = pd.to_datetime(row['生年月日'])
        rd = pd.to_datetime(row['受診日'])
        age = rd.year - bd.year
        if (rd.month, rd.day) < (bd.month, bd.day):
            age -= 1
        return age
    except:
        return None

meibo_add['受診時満年齢'] = meibo_add.apply(calc_age, axis=1)

for col in TARGET_COLS:
    if col not in meibo_add.columns:
        meibo_add[col] = None
meibo_add = meibo_add[TARGET_COLS]

# 既存データと結合・No を振り直す
meibo_all = pd.concat([meibo_base, meibo_add], ignore_index=True)
meibo_all['No'] = range(1, len(meibo_all) + 1)

# 従業員番号の重複処理
def dedup_employee(df):
    df = df.copy().reset_index(drop=True)
    dupes_mask = df.duplicated('従業員番号', keep=False) & df['従業員番号'].notna()

    for emp_id, group in df[dupes_mask].groupby('従業員番号'):
        group_sorted = group.sort_index()
        old_rows = group_sorted.iloc[:-1]
        new_idx  = group_sorted.index[-1]

        for old_idx in old_rows.index:
            old_his = df.loc[old_idx, '履歴']
            old_opt = df.loc[old_idx, 'オプション']

            # 古い行の履歴→オプションの順に連結
            parts = []
            if pd.notna(old_his) and str(old_his) not in ('', 'None', 'nan'):
                parts.append(str(old_his))
            if pd.notna(old_opt) and str(old_opt) not in ('', 'None', 'nan'):
                parts.append(str(old_opt))

            if parts:
                cur = df.loc[new_idx, '履歴']
                if pd.notna(cur) and str(cur) not in ('', 'None', 'nan'):
                    parts.insert(0, str(cur))
                df.loc[new_idx, '履歴'] = '/'.join(parts)

    df = df.drop_duplicates('従業員番号', keep='last').reset_index(drop=True)
    df['No'] = range(1, len(df) + 1)
    return df


    # 古い重複行を削除し No を振り直す
    df = df.drop_duplicates('従業員番号', keep='last').reset_index(drop=True)
    df['No'] = range(1, len(df) + 1)
    return df

meibo_all = dedup_employee(meibo_all)
print(f'📋 重複処理後: {len(meibo_all)} 件')


print(f'\n📋 合計: {len(meibo_all)} 件（追加: {len(meibo_add)} 件）')
display(meibo_all)

# 保存・ダウンロード
output_path = '健康診断名簿_統合.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl', datetime_format='YYYY/MM/DD') as writer:
    meibo_all.to_excel(writer, index=False, sheet_name='名簿')
apply_styles(output_path)
files.download(output_path)
print(f'💾 {output_path} をダウンロードしました')

既存データ: 1248 件


NameError: name 'read_meibo' is not defined